# Securing Couchbase MCP Server with Microsoft Entra ID — Non-DCR flow

This tutorial showcases how to set-up the Couchbase MCP server in Streamable HTTP mode with OAuth settings so that it serves clients that operate in the Browser flow (Non-DCR) using [Microsoft Entra ID](https://www.microsoft.com/en-us/security/business/identity-access/microsoft-entra-id) as the identity provider — a pre-registered client driving a full browser-based authorization code + PKCE flow, tested with MCP Inspector.

> ℹ️ **What is Non-DCR?** Non-DCR (non-Dynamic Client Registration) means the OAuth client app is manually registered in Entra before connecting. The client ID is provided upfront, and Entra handles the browser login and token issuance. This is in contrast to DCR ([RFC 7591](https://datatracker.ietf.org/doc/html/rfc7591)), where the client registers itself automatically — Entra does not support open DCR, so Non-DCR is the only browser-based flow available here.

> 📖 You can read more about [Couchbase MCP Server OAuth Authentication](https://mcp-server.couchbase.com/configuration/oauth) and the [Microsoft Entra ID documentation](https://learn.microsoft.com/en-us/entra/identity-platform/).

## Prerequisites

- A Microsoft Entra ID tenant with admin access
- `uvx` installed for running the MCP server
- MCP Inspector (`npx @modelcontextprotocol/inspector`) or an IDE with Non-DCR support
- A running Couchbase cluster with credentials

## What to expect

By the end of this tutorial you'll have:

- Couchbase MCP Server registered in Entra as a resource server with custom scopes
- Couchbase MCP Client registered in Entra as a public client with SPA and Mobile/Desktop platforms
- The MCP server running with PRM (Protected Resource Metadata) enabled
- A verified working connection via MCP Inspector with browser-based login

This tutorial is organized into two steps:

**Step 1 — Entra setup**

- Register the MCP server as a resource server, setting its Application ID URI to the server's own URL
- Define the `couchbase-mcp:read` and `couchbase-mcp:write` custom scopes
- Register the client application with both SPA and Mobile/Desktop redirect platforms
- Grant and consent to the delegated API permissions

**Step 2 — Non-DCR browser login flow**

- Start the MCP server with OAuth and PRM enabled so clients can discover Entra automatically
- Validate the connection using MCP Inspector

---

## Step 1 — Entra Setup

### Step 1.1 — Register the MCP Server application

This app registration represents the **resource server** — the Couchbase MCP server itself. It defines what scopes the server exposes and what the token's `aud` claim should be. See [Register an application](https://learn.microsoft.com/en-us/entra/identity-platform/quickstart-register-app) for more.

- Go to [Entra admin center](https://entra.microsoft.com/) → **App registrations** → **New registration**
- Name it `Couchbase MCP Server` (in the image below i have named it Couchbase MCP Test Server)
- Leave redirect URI blank (resource servers don't need one) → **Register**

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_entra_nondcr/entra_screenshots/Entra_2.png?raw=true" width="500">

### Step 1.2 — Set the Application ID URI

The **Application ID URI** is the globally unique identifier other apps use to ask for this API's scopes. It's what turns a bare scope name like `couchbase-mcp:read` into the fully-qualified scope a client actually requests (`<Application ID URI>/couchbase-mcp:read`, as you'll see in Step 2.2), and it's the value Entra matches against the `resource` parameter MCP clients send. See [Application ID URI](https://learn.microsoft.com/en-us/entra/identity-platform/security-best-practices-for-app-registration#application-id-uri) for more.

- In the `Couchbase MCP Server` app, go to **Expose an API**
- Click **Add** next to **Application ID URI**
- Instead of accepting the default `api://{client_id}`, set it to:

  ```
  http://127.0.0.1:8000/mcp
  ```

  This must exactly match the URL your MCP server will be running at, including the `/mcp` path — the server's PRM document advertises this exact value, and Entra requires the Application ID URI to match it.
- Click **Save**

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_entra_nondcr/entra_screenshots/Entra_4.png?raw=true" width="500">

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_entra_nondcr/entra_screenshots/Entra_5.png?raw=true" width="500">

> ⚠️ **Why not `api://...`?** The MCP server's auto-generated Protected Resource Metadata advertises its own HTTP URL as the `resource`. These two must match exactly, or you'll hit `AADSTS9010010: resource parameter doesn't match requested scopes`. Using the literal server URL sidesteps this entirely.

### Step 1.3 — Define the scopes

The Couchbase MCP server uses two scopes to enforce per-tool access control — `couchbase-mcp:read` for read-only tools and `couchbase-mcp:write` for mutation tools.

- Still in **Expose an API**, click **+ Add a scope**
- Create:
  - **Scope name**: `couchbase-mcp:read`
  - **Who can consent**: Admins and users
  - **Admin/user consent display name & description**: e.g. "Read Couchbase MCP data"
  - Click **Add scope**
- Repeat for `couchbase-mcp:write`

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_entra_nondcr/entra_screenshots/Entra_7.png?raw=true" width="500">

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_entra_nondcr/entra_screenshots/Entra_8.png?raw=true" width="500">

### Step 1.4 — Set the access token version

The access token version determines what Entra puts in the `aud` claim — and the MCP server validates `aud` on every request, so this needs to be explicit rather than left at the tenant default.

- Go to **Manifest**
- Find `"requestedAccessTokenVersion"` and change `null` to `2`
- **Save**

This ensures Entra issues v2 access tokens. v2.0 access tokens always carry the resource's client ID (GUID) in `aud`, which is why `--oauth-audience` takes the Server Application (Client) ID rather than the Application ID URI. (v1.0 tokens put the Application ID URI there instead — one reason to pin the version explicitly.) See [access token claims](https://learn.microsoft.com/en-us/entra/identity-platform/access-token-claims-reference) for the full claim reference.

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_entra_nondcr/entra_screenshots/Entra_9.png?raw=true" width="500">

### Step 1.5 — Register the Client application

This second registration is the **OAuth client** — the application that requests a token and calls the MCP server with it. In this tutorial that's MCP Inspector or your IDE, not the MCP server itself: the server registration from Step 1.1 only *declares* the scopes, while this one is what actually asks for them and receives the token. Entra keeps them separate so the same API can be called by several apps, each with its own redirect URIs and its own set of granted permissions.

It's a **public client** — it runs on the user's own machine (a browser tab or a desktop editor), so it can't keep a client secret private. That's why no secret is created here and why the flow relies on PKCE instead: the client proves it started the request by sending a one-time verifier rather than a stored credential. See [public and confidential client apps](https://learn.microsoft.com/en-us/entra/identity-platform/msal-client-applications) for more.

- **App registrations** → **New registration**
- Name it `Couchbase MCP Client` → **Register**

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_entra_nondcr/entra_screenshots/Entra_10.png?raw=true" width="500">

### Step 1.6 — Configure authentication platforms

The client needs two platform types because browser-based and native clients use different redirect mechanisms — MCP Inspector redirects to a local browser callback, while desktop editors use a native redirect.

For **MCP Inspector** (browser-based, SPA flow):

- Go to **Authentication** → **+ Add a platform** → **Single-page application**
- Redirect URI: `http://localhost:6274/oauth/callback`
- **Configure**

For other IDEs (native/desktop flow):

- **+ Add a platform** → **Mobile and desktop applications**
- Add the redirect URIs needed by the IDE
- **Configure**

> ⚠️ A redirect URI value can only exist under one platform type at a time.

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_entra_nondcr/entra_screenshots/Entra_11.png?raw=true" width="500">

### Step 1.7 — Grant API permissions

Two sets of delegated permissions are needed: the custom Couchbase MCP scopes, and the standard OpenID permissions that make browser sign-in work.

**Add the Couchbase MCP scopes:**

- Go to **API permissions** → **+ Add a permission**
- Click **My APIs** → select `Couchbase MCP Server`
- Choose **Delegated permissions** — the app acts on behalf of the signed-in user, which is what a browser login flow does. (**Application permissions** are for daemons running with no user present; see [delegated vs application permissions](https://learn.microsoft.com/en-us/entra/identity-platform/permissions-consent-overview#types-of-permissions).)
- Check both `couchbase-mcp:read` and `couchbase-mcp:write`
- **Add permissions**

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_entra_nondcr/entra_screenshots/Entra_12.png?raw=true" width="500">

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_entra_nondcr/entra_screenshots/Entra_13.png?raw=true" width="500">

**Add the Microsoft Graph OpenID permissions:**

- **+ Add a permission** again → **Microsoft Graph** → **Delegated permissions**
- Under **OpenId permissions**, check `openid`, `profile`, `email`, and `offline_access`
- **Update permissions**

> ℹ️ These four are what make the browser flow work: `openid` lets Entra issue an ID token for the sign-in, `offline_access` gets you a refresh token so the session survives access-token expiry, and `profile` / `email` carry the basic user claims. Without them the sign-in can fail outright or drop the session as soon as the first token expires.

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_entra_nondcr/entra_screenshots/Entra_14.png?raw=true" width="500">

Finally, grant consent for both sets:

- Click **Grant admin consent for [tenant]** → confirm
- All permissions should show a green checkmark under **Status**

### Step 1.8 — Collect your IDs

These values are used in the MCP server startup command and in the MCP client configuration:

| Value | Where to find it |
| --- | --- |
| Directory (Tenant) ID | `Couchbase MCP Server` app → **Overview** |
| Server Application (Client) ID | `Couchbase MCP Server` app → **Overview** |
| Client Application (Client) ID | `Couchbase MCP Client` app → **Overview** |

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_entra_nondcr/entra_screenshots/Entra_16.png?raw=true" width="500">

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_entra_nondcr/entra_screenshots/Entra_15.png?raw=true" width="500">

---

## Step 2 — Non-DCR Browser Login Flow

This flow uses a pre-registered public client with a browser-based authorization code + PKCE flow. The client ID is provided upfront, and Entra handles the browser login and token issuance.

### Step 2.1 — Start the MCP server with PRM

In [ ]:
uvx "couchbase-mcp-server" \
  --transport=http \
  --connection-string="<your-couchbase-connection-string>" \
  --username="<your-couchbase-username>" \
  --password="<your-couchbase-password>" \
  --read-only-mode=false \
  --oauth-jwks-uri="https://login.microsoftonline.com/<TENANT_ID>/discovery/v2.0/keys" \
  --oauth-issuer="https://login.microsoftonline.com/<TENANT_ID>/v2.0" \
  --oauth-audience="<SERVER_CLIENT_ID>" \
  --oauth-mcp-base-url="http://127.0.0.1:8000"

The addition of `--oauth-mcp-base-url` enables the PRM endpoint, which clients use to discover Entra as the authorization server.

Verify the PRM document:

In [ ]:
curl -s http://127.0.0.1:8000/.well-known/oauth-protected-resource/mcp | python3 -m json.tool

Expected response:

```json
{
  "resource": "http://127.0.0.1:8000/mcp",
  "authorization_servers": ["https://login.microsoftonline.com/<TENANT_ID>/v2.0"],
  "scopes_supported": ["couchbase-mcp:read", "couchbase-mcp:write"]
}
```

This `resource` value is exactly what must match the Application ID URI from Step 1.2 — if you change the server's host/port, you'll need to update the Application ID URI to match.

### Step 2.2 — Connect via MCP Inspector

```bash
npx @modelcontextprotocol/inspector
```

In the Inspector UI:

| Field | Value |
| --- | --- |
| Transport Type | `Streamable HTTP` |
| URL | `http://127.0.0.1:8000/mcp` |
| Client ID | Client Application (Client) ID |
| Client Secret | (leave blank — public client) |
| Redirect URL | `http://localhost:6274/oauth/callback` |
| Scope | `http://127.0.0.1:8000/mcp/couchbase-mcp:read http://127.0.0.1:8000/mcp/couchbase-mcp:write` |

> ⚠️ The scope format must be `<Application ID URI>/<scope-name>` — not the bare scope name. Since the Application ID URI is `http://127.0.0.1:8000/mcp`, each scope becomes `http://127.0.0.1:8000/mcp/couchbase-mcp:read`.

Click **Connect** → the Entra sign-in page opens → sign in → Inspector shows "Successfully authenticated with OAuth".

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_entra_nondcr/entra_screenshots/Entra_26.png?raw=true" width="500">

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_entra_nondcr/entra_screenshots/Entra_28.png?raw=true" width="500">

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_entra_nondcr/entra_screenshots/Entra_27.png?raw=true" width="500">

Verify:

- **Tools** tab → **List Tools** — tools should be visible
- Run a read tool → success
- Run a write tool → success
- **Auth** tab → confirm the token claims:
  - `aud` (audience) — the Server Application (Client) ID, matching `--oauth-audience`
  - `iss` (issuer) — `https://login.microsoftonline.com/<TENANT_ID>/v2.0`, matching `--oauth-issuer`
  - `scp` (scope) — Entra's name for the granted scopes; should list `couchbase-mcp:read` and `couchbase-mcp:write`

<img src="https://github.com/couchbase-examples/mcp-cookbook/blob/main/mcp_auth_entra_nondcr/entra_screenshots/Entra_31.png?raw=true" width="500">

---

## Summary

You've secured the Couchbase MCP server with Microsoft Entra ID using the Non-DCR browser login flow:

- The MCP server registered as a resource server whose **Application ID URI is the server's own URL** — the piece that keeps Entra's resource matching aligned with the PRM document
- Two scopes, `couchbase-mcp:read` and `couchbase-mcp:write`, granted to a public client with admin consent
- A client registration carrying both SPA and Mobile/Desktop redirect platforms, since browser-based and native clients redirect differently
- The MCP server publishing a PRM document that lets MCP clients discover Entra with no hardcoded endpoints
- The flow verified end to end through MCP Inspector

Because Entra does not support open Dynamic Client Registration, every client must be pre-registered — which also makes per-client scope control straightforward: register separate client apps (e.g. read-only vs read-write) and grant only the permissions each one should hold.

> 📖 Further reading: [Couchbase MCP Server OAuth Authentication](https://mcp-server.couchbase.com/configuration/oauth) · [Entra scopes and permissions](https://learn.microsoft.com/en-us/entra/identity-platform/scopes-oidc) · [MCP Authorization specification](https://modelcontextprotocol.io/specification/draft/basic/authorization)